In [4]:
import rasterio
import numpy as np
import os
from rasterio.enums import Resampling

# --- 📌 Localisation des fichiers Landsat ---
base_path = r"C:\Users\scolzy\Stage_SC_26\OMAN_Rasters_test\05_TELEDETECTION\01_LANDSAT"
prefix = "HLS.L30.T40QFJ.2025043T062900.v2.0"

# --- 📌 Définition des combinaisons ---
band_combinations = {
    "NaturalColor_432": (4, 3, 2),
    "FalseColor_543": (5, 4, 3),
    "Geological_753": (7, 5, 3),
    "Geological_bis_347": (3, 4, 7),
}

# --- 📌 Chargement des bandes Landsat ---
bands = {}
nodata_value = -9999.0  # Valeur nodata à corriger

for i in range(1, 8):  # Bandes 1 à 7
    file_path = os.path.join(base_path, f"{prefix}.B0{i}.TIF")
    if os.path.exists(file_path):
        with rasterio.open(file_path) as src:
            band_data = src.read(1, resampling=Resampling.nearest).astype(np.float32)
            
            # Remplacement des valeurs nodata
            band_data[band_data == nodata_value] = 0  # Ou np.nan si tu veux ignorer ces valeurs
            
            bands[i] = band_data
            profile = src.profile  # Métadonnées pour l'export
    else:
        print(f"⚠️ Fichier manquant : {file_path}")

# --- 📌 Génération des GeoTIFF ---
for name, (r, g, b) in band_combinations.items():
    if r in bands and g in bands and b in bands:
        rgb_image = np.stack([bands[r], bands[g], bands[b]], axis=0)

        # Normalisation des valeurs entre 0 et 65535 (uint16)
        min_val, max_val = np.nanmin(rgb_image), np.nanmax(rgb_image)
        rgb_image = ((rgb_image - min_val) / (max_val - min_val) * 65535).astype(np.uint16)

        # Mise à jour du profil pour 3 bandes
        profile.update(count=3, dtype=rasterio.uint16, nodata=0)

        output_file = os.path.join(base_path, f"{prefix}_{name}.tif")
        with rasterio.open(output_file, "w", **profile) as dst:
            dst.write(rgb_image)

        print(f"✅ Image {output_file} générée ({r}-{g}-{b})")
    else:
        print(f"❌ Impossible de générer {name}, une ou plusieurs bandes sont manquantes.")

print("🎉 Traitement terminé !")


⚠️ Fichier manquant : C:\Users\scolzy\Stage_SC_26\OMAN_Rasters_test\05_TELEDETECTION\01_LANDSAT\HLS.L30.T40QFJ.2025043T062900.v2.0.B06.TIF
✅ Image C:\Users\scolzy\Stage_SC_26\OMAN_Rasters_test\05_TELEDETECTION\01_LANDSAT\HLS.L30.T40QFJ.2025043T062900.v2.0_NaturalColor_432.tif générée (4-3-2)
✅ Image C:\Users\scolzy\Stage_SC_26\OMAN_Rasters_test\05_TELEDETECTION\01_LANDSAT\HLS.L30.T40QFJ.2025043T062900.v2.0_FalseColor_543.tif générée (5-4-3)
✅ Image C:\Users\scolzy\Stage_SC_26\OMAN_Rasters_test\05_TELEDETECTION\01_LANDSAT\HLS.L30.T40QFJ.2025043T062900.v2.0_Geological_753.tif générée (7-5-3)


RasterioIOError: Attempt to create new tiff file 'C:\Users\scolzy\Stage_SC_26\OMAN_Rasters_test\05_TELEDETECTION\01_LANDSAT\HLS.L30.T40QFJ.2025043T062900.v2.0_Geological_bis_347.tif' failed

In [13]:
import rasterio                          # Librairie pour lire/écrire des fichiers raster géospatiaux (GeoTIFF)
import numpy as np                       # Librairie pour les calculs numériques sur des tableaux
import os                                # Librairie pour manipuler les chemins de fichiers
from rasterio.enums import Resampling    # Import de la méthode de rééchantillonnage pour la lecture des bandes

# Chemin du dossier contenant les fichiers Landsat
#base_path = r"C:\Users\scolzy\Stage_SC_26\OMAN_Rasters_test\05_TELEDETECTION\01_LANDSAT"
# Nom commun à tous les fichiers de la scène Landsat (préfixe)
#prefix = "HLS.L30.T40QFJ.2025043T062900.v2.0"

# Chemin du dossier contenant les fichiers Landsat
base_path = r"C:\Users\scolzy\Stage_SC_26\BAN_Rasters_test\03_SECTEUR_TEST_EMPRISE"
# Nom commun à tous les fichiers de la scène Landsat (préfixe)
prefix = "BAN_EMPRISE_SECTEUR_TEST_LANDSAT8"

# Dictionnaire des combinaisons de bandes à générer : nom -> (rouge, vert, bleu)
band_combinations = {
    "NaturalColor_432": (4, 3, 2),      # Couleurs naturelles : Rouge=B4, Vert=B3, Bleu=B2
    "FalseColor_543": (5, 4, 3),        # Fausses couleurs : met en valeur la végétation
    "Geological_753": (7, 5, 3),        # Composition géologique classique
    "Geological_bis_347": (3, 4, 7),    # Variante géologique
}

bands = {}                              # Dictionnaire vide qui va stocker les données de chaque bande
nodata_value = -9999.0                  # Valeur utilisée dans les fichiers pour indiquer "pas de données"

for i in range(1, 8):                   # Boucle sur les bandes 1 à 7
    file_path = os.path.join(base_path, f"{prefix}_B{i}_FUSION_2024_EM.tif")  # Construction du chemin du fichier de la bande i
    if os.path.exists(file_path):       # Vérifie si le fichier existe bien sur le disque
        with rasterio.open(file_path) as src:                    # Ouvre le fichier raster
            band_data = src.read(1, resampling=Resampling.nearest).astype(np.float32)  # Lit la bande en float32
            band_data[band_data == nodata_value] = 0             # Remplace les valeurs nodata (-9999) par 0
            bands[i] = band_data                                 # Stocke la bande dans le dictionnaire
            profile = src.profile                                # Sauvegarde les métadonnées géospatiales du fichier
    else:
        print(f"⚠️ Fichier manquant : {file_path}")              # Avertit si le fichier est absent

for name, (r, g, b) in band_combinations.items():               # Boucle sur chaque combinaison de bandes
    if r in bands and g in bands and b in bands:                 # Vérifie que les 3 bandes nécessaires sont chargées
        rgb_image = np.stack([bands[r], bands[g], bands[b]], axis=0)  # Empile les 3 bandes en un tableau 3D (3, lignes, colonnes)

        #min_val, max_val = np.nanmin(rgb_image), np.nanmax(rgb_image)          # Trouve les valeurs min et max (en ignorant les NaN)
        #rgb_image = ((rgb_image - min_val) / (max_val - min_val) * 65535).astype(np.uint16)  # Normalise entre 0 et 65535 (format uint16)
        
        normalized = np.zeros_like(rgb_image)
        for idx in range(3):
            band = rgb_image[idx]
            bmin, bmax = np.nanpercentile(band, 2), np.nanpercentile(band, 98)  # percentile 2-98 pour éviter les valeurs extrêmes
            normalized[idx] = np.clip((band - bmin) / (bmax - bmin) * 65535, 0, 65535)
        rgb_image = normalized.astype(np.uint16)


        profile.update(count=3, dtype=rasterio.uint16, nodata=0)               # Met à jour le profil : 3 bandes, type uint16, nodata=0

        output_file = os.path.join(base_path, f"{prefix}_{name}.tif")          # Construit le chemin du fichier de sortie
        with rasterio.open(output_file, "w", **profile) as dst:                # Crée le fichier GeoTIFF en écriture
            dst.write(rgb_image)                                                # Écrit les 3 bandes dans le fichier

        print(f"✅ Image {output_file} générée ({r}-{g}-{b})")                 # Confirme la création du fichier
    else:
        print(f"❌ Impossible de générer {name}, une ou plusieurs bandes sont manquantes.")  # Erreur si une bande manque

print("🎉 Traitement terminé !")        # Message de fin de traitement


✅ Image C:\Users\scolzy\Stage_SC_26\BAN_Rasters_test\03_SECTEUR_TEST_EMPRISE\BAN_EMPRISE_SECTEUR_TEST_LANDSAT8_NaturalColor_432.tif générée (4-3-2)
✅ Image C:\Users\scolzy\Stage_SC_26\BAN_Rasters_test\03_SECTEUR_TEST_EMPRISE\BAN_EMPRISE_SECTEUR_TEST_LANDSAT8_FalseColor_543.tif générée (5-4-3)
✅ Image C:\Users\scolzy\Stage_SC_26\BAN_Rasters_test\03_SECTEUR_TEST_EMPRISE\BAN_EMPRISE_SECTEUR_TEST_LANDSAT8_Geological_753.tif générée (7-5-3)
✅ Image C:\Users\scolzy\Stage_SC_26\BAN_Rasters_test\03_SECTEUR_TEST_EMPRISE\BAN_EMPRISE_SECTEUR_TEST_LANDSAT8_Geological_bis_347.tif générée (3-4-7)
🎉 Traitement terminé !


In [7]:
import rasterio                          # Librairie pour lire/écrire des fichiers raster géospatiaux (GeoTIFF)
import numpy as np                       # Librairie pour les calculs numériques sur des tableaux
import os                                # Librairie pour manipuler les chemins de fichiers
from rasterio.enums import Resampling    # Import de la méthode de rééchantillonnage pour la lecture des bandes

# Chemin du dossier contenant les fichiers Landsat
base_path = r"C:\Users\scolzy\Stage_SC_26\BAN_Rasters_test\03_SECTEUR_TEST_EMPRISE"
# Nom commun à tous les fichiers de la scène Landsat (préfixe)
prefix = "BAN_EMPRISE_SECTEUR_TEST_LANDSAT8"

# --- Ratios de bandes (single-band float32 GeoTIFF) ---
# Format : "nom": (numérateur, dénominateur)
band_ratios = {
    "FerricOxydes_42": (4, 2),      # Indice d'oxydes de fer B4 (Rouge) / B2 (Bleu)
    "Clays_hydroxyles_65": (6,5),   # Indice minéraux argileux et hydroxylés B6 / B5
    "Clays_hydroxyles_75": (7,5),   # Indice minéraux argileux et hydroxylés B7 / B5
    "IOI_43": (4, 3),               # Indice d'oxyde de fer B4 (Rouge) / B3 (Vert)
    "Si_31": (3,1),                 # Indice de silice B3 (Vert) / B1 (Coastal)
    "CI_63": (6,3),                 # Indice de minéraux carbonatés B6 / B3
    "Ratio_54": (5, 4),              # Indice de rapport de bandes B5 / B4
    "Ratio_67": (6, 7)              # Indice de rapport de bandes B6 / B7
}

bands = {}                              # Dictionnaire vide qui va stocker les données de chaque bande
nodata_value = -9999.0                  # Valeur utilisée dans les fichiers pour indiquer "pas de données"

for i in range(1, 8):                   # Boucle sur les bandes 1 à 7
    file_path = os.path.join(base_path, f"{prefix}_B{i}_FUSION_2024_EM.tif")  # Construction du chemin du fichier de la bande i
    if os.path.exists(file_path):       # Vérifie si le fichier existe bien sur le disque
        with rasterio.open(file_path) as src:                    # Ouvre le fichier raster
            band_data = src.read(1, resampling=Resampling.nearest).astype(np.float32)  # Lit la bande en float32
            band_data[band_data == nodata_value] = 0             # Remplace les valeurs nodata (-9999) par 0
            bands[i] = band_data                                 # Stocke la bande dans le dictionnaire
            profile = src.profile                                # Sauvegarde les métadonnées géospatiales du fichier
    else:
        print(f"⚠️ Fichier manquant : {file_path}")              # Avertit si le fichier est absent

for name, (num, den) in band_ratios.items():             # Boucle sur chaque ratio défini
    if num in bands and den in bands:                    # Vérifie que les 2 bandes nécessaires sont chargées
        with np.errstate(divide="ignore", invalid="ignore"):  # Supprime les warnings de division par zéro
            ratio = np.where(bands[den] != 0, bands[num] / bands[den], np.nan)  # Calcule le ratio, NaN si dénominateur = 0

        min_val, max_val = np.nanmin(ratio), np.nanmax(ratio)
        ratio = ((ratio - min_val) / (max_val - min_val) * 65535).astype(np.uint16)

        profile_ratio = profile.copy()                   # Copie le profil pour ne pas modifier l'original
        profile_ratio.update(count=1, dtype=rasterio.float32, nodata=np.nan)  # 1 bande, float32, nodata=NaN

        output_file = os.path.join(base_path, f"{prefix}_{name}.tif")         # Chemin du fichier de sortie
        with rasterio.open(output_file, "w", **profile_ratio) as dst:         # Crée le GeoTIFF en écriture
            dst.write(ratio.astype(np.float32), 1)                            # Écrit la bande unique

        print(f"✅ Ratio {output_file} généré (B{num}/B{den})")               # Confirme la création
    else:
        print(f"❌ Ratio {name} impossible, bande manquante.")                # Erreur si une bande manque

print("🎉 Ratios terminés !")


✅ Ratio C:\Users\scolzy\Stage_SC_26\BAN_Rasters_test\03_SECTEUR_TEST_EMPRISE\BAN_EMPRISE_SECTEUR_TEST_LANDSAT8_FerricOxydes_42.tif généré (B4/B2)
✅ Ratio C:\Users\scolzy\Stage_SC_26\BAN_Rasters_test\03_SECTEUR_TEST_EMPRISE\BAN_EMPRISE_SECTEUR_TEST_LANDSAT8_Clays_hydroxyles_65.tif généré (B6/B5)
✅ Ratio C:\Users\scolzy\Stage_SC_26\BAN_Rasters_test\03_SECTEUR_TEST_EMPRISE\BAN_EMPRISE_SECTEUR_TEST_LANDSAT8_Clays_hydroxyles_75.tif généré (B7/B5)
✅ Ratio C:\Users\scolzy\Stage_SC_26\BAN_Rasters_test\03_SECTEUR_TEST_EMPRISE\BAN_EMPRISE_SECTEUR_TEST_LANDSAT8_IOI_43.tif généré (B4/B3)
✅ Ratio C:\Users\scolzy\Stage_SC_26\BAN_Rasters_test\03_SECTEUR_TEST_EMPRISE\BAN_EMPRISE_SECTEUR_TEST_LANDSAT8_Si_31.tif généré (B3/B1)
✅ Ratio C:\Users\scolzy\Stage_SC_26\BAN_Rasters_test\03_SECTEUR_TEST_EMPRISE\BAN_EMPRISE_SECTEUR_TEST_LANDSAT8_CI_63.tif généré (B6/B3)
🎉 Ratios terminés !


In [ ]:
import rasterio                          # Librairie pour lire/écrire des fichiers raster géospatiaux (GeoTIFF)
import numpy as np                       # Librairie pour les calculs numériques sur des tableaux
import os                                # Librairie pour manipuler les chemins de fichiers
from rasterio.enums import Resampling    # Import de la méthode de rééchantillonnage pour la lecture des bandes

# Chemin du dossier contenant les fichiers Landsat
base_path = r"C:\Users\scolzy\Stage_SC_26\BAN_Rasters_test\03_SECTEUR_TEST_EMPRISE"
# Nom commun à tous les fichiers de la scène Landsat (préfixe)
prefix = "BAN_EMPRISE_SECTEUR_TEST_LANDSAT8"

# --- Ratios de bandes (single-band float32 GeoTIFF) ---
# Format : "nom": (numérateur, dénominateur)
band_ratios = {
    "FerricOxydes_42": (4, 2),      # Indice d'oxydes de fer B4 (Rouge) / B2 (Bleu)
    "Clays_hydroxyles_65": (6,5),   # Indice minéraux argileux et hydroxylés B6 / B5
    "Clays_hydroxyles_75": (7,5),   # Indice minéraux argileux et hydroxylés B7 / B5
    "IOI_43": (4, 3),               # Indice d'oxyde de fer B4 (Rouge) / B3 (Vert)
    "Si_31": (3,1),                 # Indice de silice B3 (Vert) / B1 (Coastal)
    "CI_63": (6,3),                 # Indice de minéraux carbonatés B6 / B3
    "Ratio_54": (5, 4),              # Indice de rapport de bandes B5 / B4
    "Ratio_67": (6, 7)              # Indice de rapport de bandes B6 / B7
}

bands = {}                              # Dictionnaire vide qui va stocker les données de chaque bande
nodata_value = -9999.0                  # Valeur utilisée dans les fichiers pour indiquer "pas de données"

for i in range(1, 8):                   # Boucle sur les bandes 1 à 7
    file_path = os.path.join(base_path, f"{prefix}_B{i}_FUSION_2024_EM.tif")  # Construction du chemin du fichier de la bande i
    if os.path.exists(file_path):       # Vérifie si le fichier existe bien sur le disque
        with rasterio.open(file_path) as src:                    # Ouvre le fichier raster
            band_data = src.read(1, resampling=Resampling.nearest).astype(np.float32)  # Lit la bande en float32
            band_data[band_data == nodata_value] = 0             # Remplace les valeurs nodata (-9999) par 0
            bands[i] = band_data                                 # Stocke la bande dans le dictionnaire
            profile = src.profile                                # Sauvegarde les métadonnées géospatiales du fichier
    else:
        print(f"⚠️ Fichier manquant : {file_path}")              # Avertit si le fichier est absent

fi2 = np.where(bands[5] != 0 and bands[2] != 0, (bands[4]/bands[2])*(bands[4]+bands[6])/bands[5], np.nan)

profile_ratio = profile.copy()                                        # Copie le profil pour ne pas modifier l'original
profile_ratio.update(count=1, dtype=rasterio.float32, nodata=np.nan)  # 1 bande, float32, nodata=NaN

output_file = os.path.join(base_path, f"{prefix}_FI2.tif")         # Chemin du fichier de sortie
with rasterio.open(output_file, "w", **profile_ratio) as dst:         # Crée le GeoTIFF en écriture
    dst.write(fi2.astype(np.float32), 1)                            # Écrit la bande unique     

print("🎉 Ratios terminés !")


✅ Ratio C:\Users\scolzy\Stage_SC_26\BAN_Rasters_test\03_SECTEUR_TEST_EMPRISE\BAN_EMPRISE_SECTEUR_TEST_LANDSAT8_FerricOxydes_42.tif généré (B4/B2)
✅ Ratio C:\Users\scolzy\Stage_SC_26\BAN_Rasters_test\03_SECTEUR_TEST_EMPRISE\BAN_EMPRISE_SECTEUR_TEST_LANDSAT8_Clays_hydroxyles_65.tif généré (B6/B5)
✅ Ratio C:\Users\scolzy\Stage_SC_26\BAN_Rasters_test\03_SECTEUR_TEST_EMPRISE\BAN_EMPRISE_SECTEUR_TEST_LANDSAT8_Clays_hydroxyles_75.tif généré (B7/B5)
✅ Ratio C:\Users\scolzy\Stage_SC_26\BAN_Rasters_test\03_SECTEUR_TEST_EMPRISE\BAN_EMPRISE_SECTEUR_TEST_LANDSAT8_IOI_43.tif généré (B4/B3)
✅ Ratio C:\Users\scolzy\Stage_SC_26\BAN_Rasters_test\03_SECTEUR_TEST_EMPRISE\BAN_EMPRISE_SECTEUR_TEST_LANDSAT8_Si_31.tif généré (B3/B1)
✅ Ratio C:\Users\scolzy\Stage_SC_26\BAN_Rasters_test\03_SECTEUR_TEST_EMPRISE\BAN_EMPRISE_SECTEUR_TEST_LANDSAT8_CI_63.tif généré (B6/B3)
🎉 Ratios terminés !


In [11]:
# Chemin du dossier contenant les fichiers Landsat
base_path = r"C:\Users\scolzy\Stage_SC_26\BAN_Rasters_test\03_SECTEUR_TEST_EMPRISE"
# Nom commun à tous les fichiers de la scène Landsat (préfixe)
prefix = "BAN_EMPRISE_SECTEUR_TEST_LANDSAT8"


with rasterio.open(os.path.join(base_path, f"{prefix}_Clays_hydroxyles_75.tif")) as source:
    print(f"--- Informations pour : {source.name} ---")
    
    # 1. Dimensions (nombre de lignes et colonnes)
    print(f"Dimensions : {source.width} x {source.height}")
    
    # 2. Nombre de bandes (généralement 1 pour un MNT)
    print(f"Nombre de bandes : {source.count}")
    
    # 3. Système de Coordonnées de Référence (CRS)
    print(f"CRS : {source.crs}")
    
    # 4. Emprise spatiale (Bounding Box)
    print(f"Emprise : {source.bounds}")
    
    # 5. Résolution (taille d'un pixel en unités de la carte)
    # Souvent (30.0, 30.0) ou (1.0, 1.0)
    print(f"Résolution : {source.res}")
    
    # 6. Transformée affine (le lien entre pixels et coordonnées)
    # Permet de calculer la coordonnée X,Y de n'importe quel pixel
    print(f"Transform : \n{source.transform}")

    # Lecture de la première bande (index 1)
    # .read(1) renvoie un tableau numpy
    data = source.read(1, masked=True) 

    print(f"\n--- Valeurs ---")
    print(f"Min : {data.min():.2f} ")
    print(f"Max : {data.max():.2f} ")
    print(f"Moyenne : {data.mean():.2f} ")
    print(f"Valeur NoData : {source.nodata}")
    print(f"Type de données : {source.dtypes[0]}")

--- Informations pour : C:\Users\scolzy\Stage_SC_26\BAN_Rasters_test\03_SECTEUR_TEST_EMPRISE\BAN_EMPRISE_SECTEUR_TEST_LANDSAT8_Clays_hydroxyles_75.tif ---
Dimensions : 3019 x 3994
Nombre de bandes : 1
CRS : EPSG:32638
Emprise : BoundingBox(left=333945.0, bottom=1918035.0, right=424515.0, top=2037855.0)
Résolution : (30.0, 30.0)
Transform : 
| 30.00, 0.00, 333945.00|
| 0.00,-30.00, 2037855.00|
| 0.00, 0.00, 1.00|

--- Valeurs ---
Min : 0.00 
Max : 65535.00 
Moyenne : 18556.11 
Valeur NoData : nan
Type de données : float32
